# Smoke test: /init, /chat, URL guardrail

Start the server first:
```
python main.py
```
Requires the GPU retrieval repo to be running at `RETRIEVAL_BASE_URL` (default `http://localhost:9000`).

In [2]:
import json
import requests

BASE_URL = "http://localhost:8000"
USER_ID = "test-user"

# Two YouTube URLs to ingest. Replace with anything you want to analyze.
VIDEO_URLS = [
    "https://www.youtube.com/watch?v=uoXta0S4S3A&feature=youtu.be",
    "https://www.youtube.com/watch?v=3HajWM3xu6w&feature=youtu.be",
]

## 1. Health check

In [ ]:
r = requests.get(f"{BASE_URL}/health")
print(r.status_code, r.json())

## 2. /init — the ONLY way to start a session

Triggers ingestion on the GPU repo, extracts `video_ids`, creates an ADK session with those IDs locked into state, and returns `session_id`.

Note: ingestion can take 30 - 120 seconds depending on transcript length and whether the videos are cold.

In [3]:
payload = {"user_id": USER_ID, "urls": VIDEO_URLS}
r = requests.post(f"{BASE_URL}/init", json=payload, timeout=360)
print(r.status_code)
init_data = r.json()
print(json.dumps(init_data, indent=2))

SESSION_ID = init_data["session_id"]
VIDEO_IDS = init_data["video_ids"]
print("\nsession_id =", SESSION_ID)
print("video_ids  =", VIDEO_IDS)

200
{
  "session_id": "65910d66-6c17-47f2-af2f-2ea0652f2b48",
  "video_ids": [
    "uoXta0S4S3A",
    "3HajWM3xu6w"
  ],
  "titles": {
    "uoXta0S4S3A": "USA LO NA GRADUATION!!! \ud83c\udf93\ud83e\udd42 | A Day I\u2019ll Never Forget \u2764\ufe0f",
    "3HajWM3xu6w": "Why IRCTC Will Never Be Fixed"
  }
}

session_id = 65910d66-6c17-47f2-af2f-2ea0652f2b48
video_ids  = ['uoXta0S4S3A', '3HajWM3xu6w']


## 3. /chat — small talk

Root agent should answer directly without invoking the pipeline.

In [4]:
payload = {
    "user_id": USER_ID,
    "session_id": SESSION_ID,
    "message": "Hi! What can you do?",
}
r = requests.post(f"{BASE_URL}/chat", json=payload, timeout=120)
print(r.status_code)
print(r.json()["answer"])

200
Hi! I’m the coordinator that routes questions about the two videos you uploaded to the analysis engine. I can summarize, compare, pull metadata, timestamps, sentiment, and other analytics—tell me what you’d like to explore.


## 4. /chat — video analysis

Triggers the full pipeline: Root -> RAG (planner -> retriever -> grader loop -> packer) -> Analysis (router -> specialists -> reducer) -> Final.

In [5]:
payload = {
    "user_id": USER_ID,
    "session_id": SESSION_ID,
    "message": "Compare the two videos. Which one is more engaging and why?",
}
r = requests.post(f"{BASE_URL}/chat", json=payload, timeout=180)
print(r.status_code)
data = r.json()
print("\n--- answer ---\n")
print(data["answer"])
print("\n--- analysis state (truncated) ---\n")
print(json.dumps(data["state"].get("analysis"), indent=2)[:2000])

200

--- answer ---

Short caveat: the available data is incomplete, so this is an informed but not definitive judgment.

Verdict: 3HajWM3xu6w appears more engaging based on its retrieved transcript excerpts. It uses repeated attention-grabbing reveals (e.g., IRCTC profit figures) that create surprise and importance [3HajWM3xu6w @ 06:39], abrupt tension lines that likely spike retention ("WHAT? VERIFICATION FAILED.") [3HajWM3xu6w @ 01:47], and sharp UX/emotional language ("anti-user design") that encourages reaction and sharing [3HajWM3xu6w @ 03:25]. 

Why not definitive: there are no viewer-retention, watch-time, likes/shares, or content chunks available for uoXta0S4S3A, and platform metrics are missing for both videos, so we can’t confirm this with behavioral data.

--- analysis state (truncated) ---

{
  "dimensions_used": [
    "comparison",
    "virality"
  ],
  "confidence": "low",
  "grounded": true,
  "notes": "",
  "per_video_summary": {},
  "comparison": {
    "skipped": fals

## 5. /chat — metadata-only query

Should route to `dimensions=['metadata']` only. No mini-tier specialist calls.

In [ ]:
payload = {
    "user_id": USER_ID,
    "session_id": SESSION_ID,
    "message": "How many views does each video have?",
}
r = requests.post(f"{BASE_URL}/chat", json=payload, timeout=120)
print(r.status_code)
print(r.json()["answer"])

## 6. URL guardrail

Any URL in a chat message should be refused server-side without the runner being invoked.

In [ ]:
payload = {
    "user_id": USER_ID,
    "session_id": SESSION_ID,
    "message": "Also analyze https://www.youtube.com/watch?v=ANOTHER and compare.",
}
r = requests.post(f"{BASE_URL}/chat", json=payload, timeout=30)
print(r.status_code)
print(r.json()["answer"])

## 7. /chat — error path: bad session_id

In [ ]:
payload = {
    "user_id": USER_ID,
    "session_id": "not-a-real-session",
    "message": "hello",
}
r = requests.post(f"{BASE_URL}/chat", json=payload, timeout=30)
print(r.status_code, r.json())

## 8. /init — error path: only 1 URL

In [ ]:
payload = {"user_id": USER_ID, "urls": [VIDEO_URLS[0]]}
r = requests.post(f"{BASE_URL}/init", json=payload, timeout=30)
print(r.status_code, r.json())